# 01 - Construccion del dataset etiquetado

Genera el dataset de pares *(posicion, evaluacion)* que alimenta el
entrenamiento: descarga partidas de Lichess, las filtra, muestrea posiciones y
las etiqueta con Stockfish a profundidad fija.

Cubre las tareas 3.2 a 3.6 del WBS y cierra los requerimientos 1.1, 1.2, 1.3 y 2.3.

## Como funciona

El pipeline hace **dos pasadas** por cada dump:

1. **Extraccion** - recorre el dump comprimido (decenas de GB) una sola vez por
   streaming y guarda en un PGN chico solo las partidas que pasan el filtro. Un
   stream zstd no se puede rebobinar, asi que hacer esto una vez es lo que
   permite reanudar despues sin volver a descargar nada.
2. **Etiquetado** - lee ese extracto, muestrea 4 posiciones por partida (2 con
   blancas al turno y 2 con negras), las evalua con Stockfish y escribe shards
   Parquet, subiendo cada uno a Hugging Face apenas se cierra.

El progreso se guarda despues de **cada shard**, asi que una desconexion de
Colab cuesta como maximo un shard de trabajo.

> **Runtime: CPU, no GPU.** Stockfish es puro CPU y los runtimes con GPU de
> Colab traen *menos* vCPUs: elegir GPU aca es mas lento y ademas gasta cuota
> que conviene reservar para el entrenamiento.

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# Stockfish con version fija (queda registrada en cada fila del dataset).
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
from chessdl.colab import describe_runtime

runtime = describe_runtime("stockfish")
print(runtime.summary())

## 2. Verificacion del codigo (tests)

Antes de gastar horas de etiquetado conviene comprobar que el codigo hace lo que
dice. `pytest` descubre solo los archivos `tests/test_*.py` y ejecuta cada
funcion `test_*`; no hay que importar ni llamar nada a mano.

Son 176 tests y tardan unos 15 segundos. Cubren, entre otras cosas, el espejado
del tablero (enroque, captura al paso, promocion), la transformacion cp <-> valor
y su inversa, el filtro de partidas, el muestreo balanceado, y una construccion
completa del dataset contra un fixture con Stockfish real.

La salida de esta celda es la evidencia de los requerimientos de testing (3.1 y
3.2) para la memoria.

In [ ]:
!{sys.executable} -m pytest -q

Para ver el nombre de cada test en vez del resumen, o para correr un
subconjunto:

```
!{sys.executable} -m pytest tests/test_encoding.py -v     # un archivo, test por test
!{sys.executable} -m pytest -k mirror -v                  # los 3 tests del espejado del tablero
!{sys.executable} -m pytest --collect-only -q             # listar sin ejecutar
```

## 3. Persistencia

El disco de `/content` se recicla al desconectarse el runtime. Van a Drive las
dos cosas caras de regenerar:

- el **extracto filtrado** (cuesta una pasada completa sobre el dump), y
- el **estado de reanudacion** junto con las claves de deduplicacion.

Los shards terminados viven en Hugging Face, que es la copia durable real.

In [ ]:
from chessdl.colab import mount_drive

mount_drive()

## 4. Credenciales de Hugging Face

El token se lee del panel de **Secrets** de Colab (icono de la llave, a la
izquierda): crear un secreto `HF_TOKEN` con permiso de escritura y habilitarlo
para este notebook. Nunca pegar el token en una celda.

In [ ]:
from chessdl import hf
from chessdl.config import load_config

cfg = load_config()
token = hf.get_token()

print("Token encontrado:", token is not None)
print("Repositorio destino:", cfg.output.hf_repo_id)

## 5. Parametros

Se imprimen para que queden registrados en la salida del notebook: es la
evidencia de con que configuracion se genero cada version del dataset
(requerimiento 2.3).

In [ ]:
from chessdl.data.labeling import engine_version

print("Dumps               :", cfg.source.dumps)
print("ELO minimo          :", cfg.filter.min_elo)
print("Controles de tiempo :", cfg.filter.time_controls)
print("Plies minimos       :", cfg.filter.min_plies)
print("Posiciones/partida  :", cfg.sampling.positions_per_game, "(balanceadas por turno)")
print("Semilla de muestreo :", cfg.sampling.seed)
print("Profundidad SF      :", cfg.labeling.depth)
print("Workers             :", cfg.labeling.resolved_workers())
print("Normalizacion       : value = tanh(cp /", cfg.normalization.scale,
      "), recorte +-", cfg.normalization.cp_clip)
print("Partidas por shard  :", cfg.output.games_per_shard)
print()
sf_version = engine_version(cfg.labeling)
print("Motor:", sf_version)

## 6. Corrida piloto

Antes de lanzar la generacion completa conviene medir el rendimiento real: un
shard chico, sin subir nada, para saber cuantas posiciones por segundo etiqueta
este runtime y extrapolar el costo total.

In [ ]:
import time
from chessdl.data import pipeline

inicio = time.time()
piloto = pipeline.run_build(
    cfg,
    sf_version=sf_version,
    max_shards=1,
    max_games=200,     # extracto acotado para el piloto
    push=False,
    progress=True,
)
transcurrido = time.time() - inicio

print()
print(piloto.summary())
if piloto.n_positions:
    velocidad = piloto.n_positions / transcurrido
    print(f"\nVelocidad: {velocidad:.1f} posiciones/segundo")
    print(f"Estimado para 2.000.000 de posiciones: {2_000_000 / velocidad / 3600:.1f} horas")

> **Importante:** el piloto deja escrito estado de reanudacion y un extracto
> acotado a 200 partidas. Antes de la corrida completa hay que borrar ese estado,
> o el pipeline va a creer que el dump ya esta extraido entero.

La celda siguiente limpia lo que dejo el piloto.

In [ ]:
import shutil
from pathlib import Path
from chessdl.data.state import seen_keys_path

for ruta in [Path(cfg.output.state_path), seen_keys_path(cfg.output.state_path)]:
    if ruta.exists():
        ruta.unlink()
        print("Borrado:", ruta)

extractos = Path(cfg.output.extract_dir)
if extractos.exists():
    shutil.rmtree(extractos)
    print("Borrado:", extractos)

shards = Path(cfg.output.local_dir)
if shards.exists():
    shutil.rmtree(shards)
    print("Borrado:", shards)

## 7. Corrida completa

Esta celda es **reanudable**: si Colab se desconecta, se reconecta, se vuelven a
correr las celdas 1 a 4 y se ejecuta esta misma celda otra vez. Retoma desde el
shard siguiente al ultimo terminado.

`max_shards` acota cuanto se hace por sesion, para trabajar en tandas dentro del
limite de tiempo de un runtime.

In [ ]:
resumen = pipeline.run_build(
    cfg,
    sf_version=sf_version,
    max_shards=None,   # poner un numero para trabajar en tandas
    progress=True,
)

print()
print(resumen.summary())

## 8. Validacion de integridad

Los chequeos del requerimiento 3.2 sobre el dataset generado: sin posiciones
duplicadas, balance de color, rangos validos, sin nulos, ELO por encima del
umbral, FENs legales y no terminales, y coherencia de signo entre los dos puntos
de vista.

In [ ]:
from chessdl.data import schema
from chessdl.data.validate import describe_table, validate_table

tabla = schema.read_dataset(schema.shard_paths(cfg.output.local_dir))
reporte = validate_table(tabla, cfg)
print(reporte.summary())

In [ ]:
stats = describe_table(tabla)
for clave, valor in stats.items():
    print(f"{clave:>22}: {valor:,.4f}" if isinstance(valor, float) else f"{clave:>22}: {valor:,}")

### La misma validación desde la línea de comando

Equivalente al bloque anterior, por la vía que documenta el README. Devuelve
código de salida distinto de cero si algún chequeo falla, así que sirve para
cortar un pipeline automatizado.

In [ ]:
!{sys.executable} -m chessdl.scripts.validate_dataset --shards-dir {cfg.output.local_dir} --stats

## 9. Progreso acumulado

In [ ]:
from chessdl.data.state import PipelineState

estado = PipelineState.load(cfg.output.state_path)
for nombre, dump_state in estado.dumps.items():
    print(f"{nombre}: {dump_state.shards_done} shards, "
          f"{dump_state.positions_written:,} posiciones "
          f"({dump_state.games_accepted:,} partidas aceptadas de {dump_state.games_seen:,})")
print()
print(f"Total acumulado: {estado.total_positions:,} posiciones")

**Proximo paso:** `02_dataset_eda.ipynb` para el analisis exploratorio (WBS 3.5).